# Gene ↔ Phenotype Relation-Wise Merge

Merges Gene–Phenotype triples from GPKG, TARKG, Harmonizome (×5), and BioGrakn (×2);
resolves phenotype tail names via HPO and missing gene head names via NCBI;
deduplicates by `(head, relation, tail)`; and saves the result.

## 0. Configuration

In [1]:
import os
import pandas as pd

BASE_DIR  = '/storage/Arushi/090526_EvoAge/kg_formation/'
PROC_DIR  = BASE_DIR + 'processed_data/'
DB_DIR     = BASE_DIR + 'data_collection/databases_for_mapping/'

OUT_PATH  = BASE_DIR + 'processed_data_relation_wise_merge/generalised/GENE_PHENOTYPE/ALL_GENE_PHENOTYPE.parquet'

REQUIRED_COLS = [
    'head', 'relation', 'tail',
    'head_type', 'relation_type', 'tail_type',
    'kg_source',
    'head_id_is', 'tail_id_is',
    'head_detail_name', 'tail_detail_name', 'kg_type', 'species'
]

## 1. Build Lookup Dictionaries

### 1a. HPO — Phenotype tail names

In [3]:
HPO = pd.read_csv(DB_DIR + 'hpo/HPO.csv')
HPO_dict = dict(zip(HPO['id'], HPO['name']))
print(f"HPO dict: {len(HPO_dict):,} entries")

HPO dict: 19,484 entries


### 1b. Gene — NCBI and ENSEMBL

In [4]:
ENS_2NCBI = pd.read_csv(DB_DIR + 'ENSEMBL/ENSEMBLE_ID_2_NCBI_Gene_SYM.csv')
ENS_2NCBI = ENS_2NCBI[~ENS_2NCBI['name'].isna()]
NCBI_2ENS__dict = dict(zip(ENS_2NCBI['name'], ENS_2NCBI['initial_alias']))
del ENS_2NCBI

NCBI_ALL_GENE = pd.read_csv(DB_DIR + 'ncbi/Homo_sapiens.gene_info', sep='\t')
NCBI_ALL_GENE['ENSEMBLE_ID'] = NCBI_ALL_GENE['Symbol'].map(NCBI_2ENS__dict)
NCBI_ALL_GENEname_dict   = dict(zip(NCBI_ALL_GENE['GeneID'], NCBI_ALL_GENE['description']))
NCBI_ALL_GENEIDname_dict = dict(zip(NCBI_ALL_GENE['GeneID'], NCBI_ALL_GENE['Symbol']))
NCBI_ALL_Symb_Desc_dict  = dict(zip(NCBI_ALL_GENE['Symbol'], NCBI_ALL_GENE['description']))

NCBI_ALL_GENE_Syn_Dict = dict(zip(NCBI_ALL_GENE['Synonyms'], NCBI_ALL_GENE['Symbol']))
exploded_dict_NCBI_ALL_GENE_Syn_Dict = {}
for k, v in NCBI_ALL_GENE_Syn_Dict.items():
    for alias in k.split('|'):
        exploded_dict_NCBI_ALL_GENE_Syn_Dict[alias.strip()] = v

print(f"NCBI gene table: {len(NCBI_ALL_GENE):,} rows")
print(f"Synonym dict:    {len(exploded_dict_NCBI_ALL_GENE_Syn_Dict):,} entries")

NCBI gene table: 193,505 rows
Synonym dict:    69,564 entries


## 2. Load KG Sources

### 2a. GPKG

In [9]:
GPKG_Gene_Phenotype = pd.read_csv(PROC_DIR + 'GPKG/Gene_Phenotype_GPKG.csv')
GPKG_Gene_Phenotype.columns = GPKG_Gene_Phenotype.columns.str.lower()
print(f"GPKG:          {len(GPKG_Gene_Phenotype):,} rows")

GPKG_Gene_Phenotype['kg_type'] = 'Generalised'
GPKG_Gene_Phenotype['kg_source'] = 'GPKG'
GPKG_Gene_Phenotype['species'] = 'Homo species'
GPKG_Gene_Phenotype.head(2)


GPKG:          209,416 rows


,head,relation,tail,head_type,relation_type,tail_type,kg_source,head_orignal,head_alias_mapped,head_detail_name,head_id_is,tail_name,tail_id_is,kg_type,species
0,CLPP,Gene_Phenotype,HP:0000252,Gene,associated,Phenotype,GPKG,ENSG00000125656,CLPP,caseinolytic mitochondrial matrix peptidase pr...,NCBI_ID,Microcephaly,HPO_ID,Generalised,Homo species
1,CLPP,Gene_Phenotype,HP:0001250,Gene,associated,Phenotype,GPKG,ENSG00000125656,CLPP,caseinolytic mitochondrial matrix peptidase pr...,NCBI_ID,Seizure,HPO_ID,Generalised,Homo species


### 2b. TARKG

In [31]:
TARKG_Gene_Phenotype = pd.read_csv(PROC_DIR + 'TARKG/Gene_Phenotype_TARKG.csv')
TARKG_Gene_Phenotype.columns = TARKG_Gene_Phenotype.columns.str.lower()
print(f"TARKG:         {len(TARKG_Gene_Phenotype):,} rows")
TARKG_Gene_Phenotype['kg_type'] = 'Generalised'
TARKG_Gene_Phenotype['kg_source'] = 'TARKG'
TARKG_Gene_Phenotype['species'] = 'Homo species'
TARKG_Gene_Phenotype.head(2)

TARKG:         157,919 rows


,head,relation,tail,head_type,relation_type,tail_type,kg_source,head_detail_name,tail_detail_name,head_id_is,tail_id_is,kg_index,kg,change,kg_type,species
0,TP53,Gene_Phenotype,HP:0030448,Gene,GENE_PHENOTYPE,Phenotype,TARKG,tumor protein p53,Soft tissue sarcoma,NCBI_ID,HPO_ID,OpenBioLink-3740008,OpenBioLink,0,Generalised,Homo species
1,BARD1,Gene_Phenotype,HP:0000006,Gene,GENE_PHENOTYPE,Phenotype,TARKG,BRCA1 associated RING domain 1,Autosomal dominant inheritance,NCBI_ID,HPO_ID,OpenBioLink-3222864,OpenBioLink,0,Generalised,Homo species


### 2c. Harmonizome

In [32]:
harmonizome = pd.read_csv(PROC_DIR + 'harmonizome/Gene_Phenotype_Harmonizome.csv')
harmonizome.columns = harmonizome.columns.str.lower()
print(f"harmonizome:         {len(harmonizome):,} rows")
harmonizome['kg_type'] = 'Generalised'
harmonizome['kg_source'] = 'Harmonizome'
harmonizome['species'] = 'Homo species'
harmonizome

harmonizome:         594,596 rows


,head,relation,tail,head_type,tail_type,source,kg_source,head_detail_name,tail_detail_name,head_id_is,tail_id_is,kg_type,species
0,IFI16,Gene_Phenotype,HP:0010987,Gene,Phenotype,GWASdb,Harmonizome,interferon gamma inducible protein 16,Abnormal cellular immune system morphology,NCBI_ID,HPO_ID,Generalised,Homo species
1,CD84,Gene_Phenotype,HP:0010987,Gene,Phenotype,GWASdb,Harmonizome,CD84 molecule,Abnormal cellular immune system morphology,NCBI_ID,HPO_ID,Generalised,Homo species
2,FCRLA,Gene_Phenotype,HP:0010987,Gene,Phenotype,GWASdb,Harmonizome,Fc receptor like A,Abnormal cellular immune system morphology,NCBI_ID,HPO_ID,Generalised,Homo species
3,RCHY1,Gene_Phenotype,HP:0010987,Gene,Phenotype,GWASdb,Harmonizome,ring finger and CHY zinc finger domain contain...,Abnormal cellular immune system morphology,NCBI_ID,HPO_ID,Generalised,Homo species
4,DPY19L1P1,Gene_Phenotype,HP:0010987,Gene,Phenotype,GWASdb,Harmonizome,DPY19L1 pseudogene 1,Abnormal cellular immune system morphology,NCBI_ID,HPO_ID,Generalised,Homo species
...,...,...,...,...,...,...,...,...,...,...,...,...,...
594591,CXCR3,Gene_Phenotype,HP:0002102,Gene,Phenotype,HuGE Navigator,Harmonizome,C-X-C motif chemokine receptor 3,Pleuritis,NCBI_ID,HPO_ID,Generalised,Homo species
594592,STX17,Gene_Phenotype,HP:0012289,Gene,Phenotype,HuGE Navigator,Harmonizome,syntaxin 17,Facial neoplasm,NCBI_ID,HPO_ID,Generalised,Homo species
594593,TCOF1,Gene_Phenotype,HP:0005321,Gene,Phenotype,HuGE Navigator,Harmonizome,treacle ribosome biogenesis factor 1,Mandibulofacial dysostosis,NCBI_ID,HPO_ID,Generalised,Homo species
594594,RS1,Gene_Phenotype,HP:0030502,Gene,Phenotype,HuGE Navigator,Harmonizome,retinoschisin 1,Retinoschisis,NCBI_ID,HPO_ID,Generalised,Homo species


### 2d. BioGrakn 

In [33]:


BioGrakn_Gene_Phenotype_1 = pd.read_csv(PROC_DIR + 'BioGrakn/Gene_Phenotype_2_Biogrkn.csv')
BioGrakn_Gene_Phenotype_1.columns = BioGrakn_Gene_Phenotype_1.columns.str.lower()

print(f"BioGrakn 1: {len(BioGrakn_Gene_Phenotype_1):,} rows")
BioGrakn_Gene_Phenotype_1['kg_type'] = 'Generalised'

BioGrakn_Gene_Phenotype_1['species'] = 'Homo species'
BioGrakn_Gene_Phenotype_1

BioGrakn 1: 26,926 rows


,head,relation,tail,head_type,tail_type,kg_source,head_detail_name,tail_detail_name,head_id_is,tail_id_is,source,kg_type,species
0,A1BG,Gene_Phenotype,HP:0002240,Gene,Phenotype,BioGrakn,alpha-1-B glycoprotein,Hepatomegaly,NCBI_ID,HPO_ID,CTD_human,Generalised,Homo species
1,A1BG,Gene_Phenotype,HP:0100753,Gene,Phenotype,BioGrakn,alpha-1-B glycoprotein,Schizophrenia,NCBI_ID,HPO_ID,CTD_human,Generalised,Homo species
2,IGHA2,Gene_Phenotype,HP:0002511,Gene,Phenotype,BioGrakn,immunoglobulin heavy constant alpha 2 (A2m mar...,Alzheimer disease,NCBI_ID,HPO_ID,CTD_human,Generalised,Homo species
3,IGHA2,Gene_Phenotype,HP:0003003,Gene,Phenotype,BioGrakn,immunoglobulin heavy constant alpha 2 (A2m mar...,Colon cancer,NCBI_ID,HPO_ID,CTD_human,Generalised,Homo species
4,IGHA2,Gene_Phenotype,HP:0100273,Gene,Phenotype,BioGrakn,immunoglobulin heavy constant alpha 2 (A2m mar...,Neoplasm of the colon,NCBI_ID,HPO_ID,CTD_human,Generalised,Homo species
...,...,...,...,...,...,...,...,...,...,...,...,...,...
26921,OCA5,Gene_Phenotype,HP:0001107,Gene,Phenotype,BioGrakn,oculocutaneous albinism 5 (autosomal recessive),Ocular albinism,NCBI_ID,HPO_ID,GENOMICS_ENGLAND,Generalised,Homo species
26922,MIR7704,Gene_Phenotype,HP:0005978,Gene,Phenotype,BioGrakn,microRNA 7704,Type II diabetes mellitus,NCBI_ID,HPO_ID,CTD_human,Generalised,Homo species
26923,MIR8061,Gene_Phenotype,HP:0005978,Gene,Phenotype,BioGrakn,microRNA 8061,Type II diabetes mellitus,NCBI_ID,HPO_ID,CTD_human,Generalised,Homo species
26924,MIR6741,Gene_Phenotype,HP:0005978,Gene,Phenotype,BioGrakn,microRNA 6741,Type II diabetes mellitus,NCBI_ID,HPO_ID,CTD_human,Generalised,Homo species


In [43]:
### 2e. Monarch

In [45]:
Monarch = pd.read_csv(PROC_DIR + 'Monarch/Human/Gene_Human_Phenotype_MONARCH.csv')
Monarch.columns = Monarch.columns.str.lower()
Monarch.rename(columns={'tail_name': 'tail_detail_name'}, inplace=True)
print(f"BioGrakn 1: {len(Monarch):,} rows")
Monarch['kg_type'] = 'Generalised'

Monarch['species'] = 'Homo species'
Monarch

BioGrakn 1: 312,607 rows


,head,relation,tail,head_type,relation_type,tail_type,relation_source,kg_source,head_detail_name,head_taxon_name,tail_taxon_name,head_id_is,tail_id_is,head_name,tail_detail_name,head_orignal,species_species,kg_type,species
0,IFT172,Gene_PhenotypicFeature,HP:0007663,Gene,has_phenotype,PhenotypicFeature,infores:hpo-annotations,Monarch_KG,intraflagellar transport 172,Homo sapiens,NaN,NCBI_ID,HP,IFT172,Reduced visual acuity,HGNC:30391,Homo sapiens_Homo sapiens,Generalised,Homo species
1,IFT172,Gene_PhenotypicFeature,HP:0000119,Gene,has_phenotype,PhenotypicFeature,infores:hpo-annotations,Monarch_KG,intraflagellar transport 172,Homo sapiens,NaN,NCBI_ID,HP,IFT172,Abnormality of the genitourinary system,HGNC:30391,Homo sapiens_Homo sapiens,Generalised,Homo species
2,IFT172,Gene_PhenotypicFeature,HP:0000126,Gene,has_phenotype,PhenotypicFeature,infores:hpo-annotations,Monarch_KG,intraflagellar transport 172,Homo sapiens,NaN,NCBI_ID,HP,IFT172,Hydronephrosis,HGNC:30391,Homo sapiens_Homo sapiens,Generalised,Homo species
3,IFT172,Gene_PhenotypicFeature,HP:0000100,Gene,has_phenotype,PhenotypicFeature,infores:hpo-annotations,Monarch_KG,intraflagellar transport 172,Homo sapiens,NaN,NCBI_ID,HP,IFT172,Nephrotic syndrome,HGNC:30391,Homo sapiens_Homo sapiens,Generalised,Homo species
4,IFT172,Gene_PhenotypicFeature,HP:0000112,Gene,has_phenotype,PhenotypicFeature,infores:hpo-annotations,Monarch_KG,intraflagellar transport 172,Homo sapiens,NaN,NCBI_ID,HP,IFT172,Nephropathy,HGNC:30391,Homo sapiens_Homo sapiens,Generalised,Homo species
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
312602,NEUROG1,Gene_PhenotypicFeature,HP:0003390,Gene,has_phenotype,PhenotypicFeature,infores:hpo-annotations,Monarch_KG,neurogenin 1,Homo sapiens,NaN,NCBI_ID,HP,NEUROG1,Sensory axonal neuropathy,HGNC:7764,Homo sapiens_Homo sapiens,Generalised,Homo species
312603,NEUROG1,Gene_PhenotypicFeature,HP:0100716,Gene,has_phenotype,PhenotypicFeature,infores:hpo-annotations,Monarch_KG,neurogenin 1,Homo sapiens,NaN,NCBI_ID,HP,NEUROG1,Self-injurious behavior,HGNC:7764,Homo sapiens_Homo sapiens,Generalised,Homo species
312604,NEUROG1,Gene_PhenotypicFeature,HP:0100785,Gene,has_phenotype,PhenotypicFeature,infores:hpo-annotations,Monarch_KG,neurogenin 1,Homo sapiens,NaN,NCBI_ID,HP,NEUROG1,Insomnia,HGNC:7764,Homo sapiens_Homo sapiens,Generalised,Homo species
312605,NEUROG1,Gene_PhenotypicFeature,HP:0011968,Gene,has_phenotype,PhenotypicFeature,infores:hpo-annotations,Monarch_KG,neurogenin 1,Homo sapiens,NaN,NCBI_ID,HP,NEUROG1,Feeding difficulties,HGNC:7764,Homo sapiens_Homo sapiens,Generalised,Homo species


## 3. Check and Fix Duplicate Columns

In [46]:
SOURCE_DFS = [
    ('GPKG_Gene_Phenotype',       GPKG_Gene_Phenotype),
    ('TARKG_Gene_Phenotype',      TARKG_Gene_Phenotype),
    ('harmonizome',             harmonizome),
    ('BioGrakn_Gene_Phenotype_1', BioGrakn_Gene_Phenotype_1),
    ('Monarch', Monarch)
]

for name, df in SOURCE_DFS:
    dupes = df.columns[df.columns.duplicated(keep=False)].unique().tolist()
    if dupes:
        print(f"\n[{name}] duplicate columns: {dupes}")
        for col in dupes:
            for i, (_, s) in enumerate(df.loc[:, df.columns == col].items()):
                print(f"  '{col}' occurrence {i} → sample: {s.dropna().head(3).tolist()}")
    else:
        print(f"[{name}] ✓ no duplicates")

[GPKG_Gene_Phenotype] ✓ no duplicates
[TARKG_Gene_Phenotype] ✓ no duplicates
[harmonizome] ✓ no duplicates
[BioGrakn_Gene_Phenotype_1] ✓ no duplicates
[Monarch] ✓ no duplicates


In [47]:
SOURCE_DFS = [(name, df.loc[:, ~df.columns.duplicated(keep='first')]) for name, df in SOURCE_DFS]

for name, df in SOURCE_DFS:
    remaining = df.columns[df.columns.duplicated()].tolist()
    print(f"[{name}] {'✓ clean' if not remaining else '✗ dupes: ' + str(remaining)}")

[GPKG_Gene_Phenotype] ✓ clean
[TARKG_Gene_Phenotype] ✓ clean
[harmonizome] ✓ clean
[BioGrakn_Gene_Phenotype_1] ✓ clean
[Monarch] ✓ clean


## 4. Align Schemas and Concatenate

In [48]:
CONCAT_COLS = [
    'head', 'relation', 'tail',
    'head_type', 'relation_type', 'tail_type',
    'kg_source', 'head_id_is', 'tail_id_is',
    'head_detail_name', 'tail_detail_name',
    'kg_type',
    'species'
]

df_list = []
for name, df in SOURCE_DFS:
    tmp = df.loc[:, ~df.columns.duplicated()].copy()
    for col in CONCAT_COLS:
        if col not in tmp.columns:
            tmp[col] = pd.NA
    df_list.append(tmp[CONCAT_COLS])

final_df = pd.concat(df_list, ignore_index=True)
print(f"Combined: {len(final_df):,} rows")
final_df.head(2)

Combined: 1,301,464 rows


,head,relation,tail,head_type,relation_type,tail_type,kg_source,head_id_is,tail_id_is,head_detail_name,tail_detail_name,kg_type,species
0,CLPP,Gene_Phenotype,HP:0000252,Gene,associated,Phenotype,GPKG,NCBI_ID,HPO_ID,caseinolytic mitochondrial matrix peptidase pr...,NaN,Generalised,Homo species
1,CLPP,Gene_Phenotype,HP:0001250,Gene,associated,Phenotype,GPKG,NCBI_ID,HPO_ID,caseinolytic mitochondrial matrix peptidase pr...,NaN,Generalised,Homo species


## 5. Standardise Fixed-Value Columns

In [49]:
final_df['head_type'] = 'Gene'
final_df['tail_type'] = 'Phenotype'
final_df['relation']  = 'Gene_Phenotype'

print("Unique relation:",   set(final_df['relation']))
print("Unique head_id_is:", set(final_df['head_id_is']))
print("Unique tail_id_is:", set(final_df['tail_id_is']))
print("Unique kg_source:",  set(final_df['kg_source']))

Unique relation: {'Gene_Phenotype'}
Unique head_id_is: {'NCBI_ID'}
Unique tail_id_is: {'HP', 'HPO_ID'}
Unique kg_source: {'Monarch_KG', 'BioGrakn', 'Harmonizome', 'TARKG', 'GPKG'}


## 6. Deduplicate

In [50]:
GROUP_COLS = ['head', 'relation', 'tail']

def merge_sources(x):
    return '::'.join(sorted(set(x.dropna())))

final_df_dedup = final_df.groupby(GROUP_COLS, as_index=False).agg({
    'head_type':        'first',
    'relation_type':    'first',
    'tail_type':        'first',
    'kg_source':         merge_sources,
    'head_id_is':       'first',
    'tail_id_is':       'first',
    'head_detail_name': 'first',
    'tail_detail_name': 'first',
    'kg_type': merge_sources,
    'species': 'first',
})
print(f"After deduplication: {len(final_df_dedup):,} rows")

After deduplication: 825,423 rows


## 7. Resolve Tail (HPO) and Head (Gene) Names

1. Map phenotype tails → HPO names.
2. Repair missing gene head names via synonym + NCBI description.
3. Drop rows still missing `head_detail_name`.

In [51]:
# Step 1 – HPO tail names
final_df_dedup['tail_detail_name'] = final_df_dedup['tail'].map(HPO_dict)

# Step 2 – repair gene head names
mask = final_df_dedup['head_detail_name'].isna()
print(f"Rows needing head_detail_name repair: {mask.sum():,}")

# Clean symbol (remove '-') then map synonyms → canonical symbol
final_df_dedup.loc[mask, 'head'] = final_df_dedup.loc[mask, 'head'].str.replace('-', '', regex=False)
final_df_dedup.loc[mask, 'head'] = (
    final_df_dedup.loc[mask, 'head']
    .map(exploded_dict_NCBI_ALL_GENE_Syn_Dict)
    .fillna(final_df_dedup.loc[mask, 'head'])
)

# NCBI Symbol → description
mask2 = final_df_dedup['head_detail_name'].isna()
final_df_dedup.loc[mask2, 'head_detail_name'] = final_df_dedup.loc[mask2, 'head'].map(NCBI_ALL_Symb_Desc_dict)
print(f"Still missing head_detail_name: {final_df_dedup['head_detail_name'].isna().sum():,}")

# Step 3 – drop rows still missing head name
before = len(final_df_dedup)
final_df_dedup = final_df_dedup[~final_df_dedup['head_detail_name'].isna()].reset_index(drop=True)
print(f"Dropped {before - len(final_df_dedup):,} rows with missing head_detail_name → {len(final_df_dedup):,} remaining")

Rows needing head_detail_name repair: 2,806
Still missing head_detail_name: 124
Dropped 124 rows with missing head_detail_name → 825,299 remaining


## 8. Add Schema Columns and Enforce Column Order

In [52]:

final_df_dedup = final_df_dedup[REQUIRED_COLS]
print(f"Final shape: {final_df_dedup.shape}")
final_df_dedup.head()

Final shape: (825299, 13)


,head,relation,tail,head_type,relation_type,tail_type,kg_source,head_id_is,tail_id_is,head_detail_name,tail_detail_name,kg_type,species
0,A1BG,Gene_Phenotype,HP:0002240,Gene,None,Phenotype,BioGrakn,NCBI_ID,HPO_ID,alpha-1-B glycoprotein,Hepatomegaly,Generalised,Homo species
1,A1BG,Gene_Phenotype,HP:0100753,Gene,None,Phenotype,BioGrakn,NCBI_ID,HPO_ID,alpha-1-B glycoprotein,Schizophrenia,Generalised,Homo species
2,A1CF,Gene_Phenotype,HP:0000001,Gene,None,Phenotype,Harmonizome,NCBI_ID,HPO_ID,APOBEC1 complementation factor,All,Generalised,Homo species
3,A1CF,Gene_Phenotype,HP:0000002,Gene,None,Phenotype,Harmonizome,NCBI_ID,HPO_ID,APOBEC1 complementation factor,Abnormality of body height,Generalised,Homo species
4,A1CF,Gene_Phenotype,HP:0000079,Gene,None,Phenotype,Harmonizome,NCBI_ID,HPO_ID,APOBEC1 complementation factor,Abnormality of the urinary system,Generalised,Homo species


## 9. QC — NaN Check

In [53]:
def nan_summary(df):
    true_nan   = df.isna().sum()
    string_nan = df.apply(lambda c: c.astype(str).str.upper().eq('NAN').sum())
    return pd.DataFrame({
        'NaN_count':          true_nan,
        "'NAN'_string_count": string_nan,
        'Total_NaN_like':     true_nan + string_nan
    })

nan_summary(final_df_dedup)

,NaN_count,'NAN'_string_count,Total_NaN_like
head,0,0,0
relation,0,0,0
tail,0,0,0
head_type,0,0,0
relation_type,530001,0,530001
tail_type,0,0,0
kg_source,0,0,0
head_id_is,0,0,0
tail_id_is,0,0,0
head_detail_name,0,0,0


## 10. Save Output

In [54]:
os.makedirs(os.path.dirname(OUT_PATH), exist_ok=True)
final_df_dedup.to_parquet(OUT_PATH, index=False)
print(f"Saved {len(final_df_dedup):,} rows → {OUT_PATH}")

Saved 825,299 rows → /storage/Arushi/090526_EvoAge/kg_formation/processed_data_relation_wise_merge/generalised/GENE_PHENOTYPE/ALL_GENE_PHENOTYPE..parquet
